# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [3]:
import pandas as pd

chunks = pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000)
first_chunk = next(chunks)
print(len(first_chunk))

# Just for testing
first_chunk.head()


100000


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A10000012B7CGYKOMPQ4L,000100039X,Adam,"[0, 0]",Spiritually and mentally inspiring! A book tha...,5,Wonderful!,1355616000,"12 16, 2012"
1,A2S166WSCFIFP5,000100039X,"adead_poet@hotmail.com ""adead_poet@hotmail.com""","[0, 2]",This is one my must have books. It is a master...,5,close to god,1071100800,"12 11, 2003"
2,A1BM81XB4QHOA3,000100039X,"Ahoro Blethends ""Seriously""","[0, 0]",This book provides a reflection that you can a...,5,Must Read for Life Afficianados,1390003200,"01 18, 2014"
3,A1MOSTXNIO5MPJ,000100039X,Alan Krug,"[0, 0]",I first read THE PROPHET in college back in th...,5,Timeless for every good and bad time in your l...,1317081600,"09 27, 2011"
4,A2XQ5LZHTD4AFT,000100039X,Alaturka,"[7, 9]",A timeless classic. It is a very demanding an...,5,A Modern Rumi,1033948800,"10 7, 2002"


In [4]:
df = first_chunk
df["user_id"] = pd.factorize(df["reviewerID"])[0]
df["item_id"] = pd.factorize(df["asin"])[0]
df["interaction"] = 1


In [5]:
df = df.sort_values(["user_id", "unixReviewTime"]).copy()

df_test = df.groupby("user_id").tail(1).copy()
df_train = df.drop(df_test.index).copy()

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# MostPopular section

In [6]:
item_popularity = (
    df_train.groupby("item_id")
    .size()
    .sort_values(ascending=False)
)

popular_items = item_popularity.index.tolist()
user_seen_train = df_train.groupby("user_id")["item_id"].apply(set).to_dict()

print(popular_items)

[591, 11, 552, 1552, 244, 601, 42, 1873, 1684, 2040, 172, 321, 367, 1310, 315, 758, 814, 391, 2607, 2447, 817, 2135, 195, 1693, 2146, 280, 13, 2351, 593, 691, 757, 292, 2328, 260, 2005, 2011, 274, 386, 2403, 1779, 190, 373, 320, 2038, 2587, 560, 2421, 2169, 2603, 1871, 2103, 457, 136, 1152, 2151, 2520, 1863, 1459, 2393, 1461, 587, 521, 39, 0, 1472, 1460, 2064, 38, 293, 2614, 1643, 1833, 1838, 437, 2375, 1416, 441, 2525, 2524, 1228, 477, 1981, 1205, 1327, 1368, 2523, 276, 2439, 277, 2522, 1158, 159, 1692, 1988, 2506, 509, 1995, 142, 2440, 180, 1581, 249, 819, 2157, 2254, 400, 165, 1058, 2012, 2554, 1178, 2325, 1900, 1332, 574, 833, 219, 2108, 1359, 294, 364, 575, 1296, 325, 2615, 1880, 2019, 415, 2576, 878, 1153, 2102, 553, 2359, 455, 1093, 1477, 446, 1443, 1832, 1361, 635, 1525, 2058, 745, 1336, 2118, 1805, 1388, 2110, 2109, 2162, 1331, 1257, 569, 241, 160, 1656, 576, 2507, 2178, 2153, 409, 2635, 69, 716, 1269, 1284, 1341, 1735, 1734, 317, 2424, 1865, 1732, 2211, 2390, 1276, 436, 2427,

In [25]:
import math


def recommend_most_popular(user_id, popular_items, user_seen_train, k=10):
    seen = user_seen_train.get(user_id, set())
    recs = []
    for item in popular_items:
        if item not in seen:
            recs.append(item)
        if len(recs) == k:
            break
    return recs

def recall_at_k(recommended_items, ground_truth_item):
    return 1.0 if ground_truth_item in recommended_items else 0.0

def precision_at_k(recommended_items, ground_truth_item,k):
    return 1.0/k if ground_truth_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, ground_truth_item):
    if ground_truth_item in recommended_items:
        rank = recommended_items.index(ground_truth_item) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0


def map(predictions, ground_truth, k=None):
    total_ap = 0
    users = 0

    for user in predictions:
        if user not in ground_truth or len(ground_truth[user]) == 0:
            continue

        ranked = predictions[user][:k] if k else predictions[user]
        relevant = ground_truth[user]

        hits = 0
        sum_precisions = 0

        for i, item in enumerate(ranked, start=1):
            if item in relevant:
                hits += 1
                sum_precisions += hits / i

        total_ap += sum_precisions / len(relevant)
        users += 1

    return total_ap / users if users > 0 else 0


def mrr(predictions, ground_truth, k=None):
    total = 0
    users = 0

    for user in predictions:
        if user not in ground_truth or len(ground_truth[user]) == 0:
            continue

        ranked = predictions[user][:k] if k else predictions[user]
        relevant = ground_truth[user]

        for rank, item in enumerate(ranked, start=1):
            if item in relevant:
                total += 1 / rank
                break

        users += 1

    return total / users if users > 0 else 0

In [27]:
K = 10
recalls = []
ndcgs = []
precisions = []
ground_truths = {}
all_recs = {}

for _, row in df_test.iterrows():
    user_id = row["user_id"]
    ground_truth_item = row["item_id"]

    recs = recommend_most_popular(user_id, popular_items, user_seen_train, k=K)
    ground_truths[user_id] = [ground_truth_item]
    all_recs[user_id] = recs
    recalls.append(recall_at_k(recs, ground_truth_item))
    precisions.append(precision_at_k(recs,ground_truth_item,K))
    ndcgs.append(ndcg_at_k(recs, ground_truth_item))

# PROBABLY NEED TO RE-WRITE THOSE FOR THE REAL THING COZ I REFUSE TO BELIEVE EACH USER WILL ACTUALLY ONLY HAVE ONE GROUND TRUTH ITEM
print(f"Users evaluated: {len(recalls)}")
# metrics to use for ranking specifically: Precision@K and Recall@K
print(f"Precision@{K}: {sum(precisions)/len(precisions):.4f}")
print(f"Recall@{K}: {sum(recalls)/len(recalls):.4f}")
# also nice for within-list ordering checks: NDCG@K, MAP AND MRR
print(f"NDCG@{K}: {sum(ndcgs)/len(ndcgs):.4f}")
print("MAP:", map(all_recs,ground_truths,K))
print("MRR:", mrr(all_recs,ground_truths,K))

Users evaluated: 68136
Precision@10: 0.0200
Recall@10: 0.2004
NDCG@10: 0.1069
MAP: 0.07814970549098915
MRR: 0.07814970549098915


## BRP